# Keywords Selection and Analysis

This notebook contains the implementation for the project tasks. Run cells in order because later tasks reuse functions defined earlier. See `README.md` for environment setup, external datasets, the word2vec model, and SEKE configuration.

In [ ]:
"""Task 1: collect biomedical papers from PubMed and save them as data/database.csv."""
import os
import time
import xml.etree.ElementTree as ET

import pandas as pd
import requests

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

# PubMed cannot filter on author keywords, so records without them are dropped later
SEARCH_TERM = (
    "(cancer OR diabetes OR cardiovascular OR neurological OR infection OR immunology OR genetics) "
    "AND hasabstract AND english[lang] AND journal article[pt] AND 2021:2025[dp]"
)

MIN_KEYWORDS = 3
MIN_ABSTRACT_WORDS = 50
KEYWORD_SEPARATOR = ";"

MY_EMAIL = os.getenv("NCBI_EMAIL", "your-email@example.com")
API_KEY = os.getenv("NCBI_API_KEY")
TARGET = 1000
POOL = 6000
OUTPUT_PATH = "data/database.csv"


def get_with_retries(url, params, tries=4):
    for attempt in range(tries):
        try:
            response = requests.get(url, params=params, timeout=60)
            response.raise_for_status()
            return response
        except requests.RequestException as error:
            print(f"  request failed ({error}), retry {attempt + 1}/{tries}")
            time.sleep(2 * (attempt + 1))
    raise RuntimeError("PubMed did not answer, try again later")


def search_ids(term, how_many, base_params):
    params = dict(base_params, db="pubmed", term=term, retmax=how_many, retmode="json")
    response = get_with_retries(EUTILS + "esearch.fcgi", params)
    return response.json()["esearchresult"]["idlist"]


def text_of(element):
    if element is None:
        return ""
    return " ".join("".join(element.itertext()).split())


def parse_articles(xml_text):
    root = ET.fromstring(xml_text)
    articles = []
    for article in root.iter("PubmedArticle"):
        pmid = text_of(article.find("MedlineCitation/PMID"))
        title = text_of(article.find("MedlineCitation/Article/ArticleTitle"))

        abstract_parts = []
        for part in article.findall("MedlineCitation/Article/Abstract/AbstractText"):
            part_text = text_of(part)
            label = part.get("Label")
            abstract_parts.append(f"{label}: {part_text}" if label else part_text)
        abstract = " ".join(abstract_parts)

        keywords = []
        for keyword in article.findall("MedlineCitation/KeywordList/Keyword"):
            word = text_of(keyword)
            already_there = [k.lower() for k in keywords]
            if word and word.lower() not in already_there:
                keywords.append(word)

        articles.append({"pmid": pmid, "title": title, "abstract": abstract, "keywords": keywords})
    return articles


def is_good_record(record):
    return bool(
        record["title"]
        and len(record["abstract"].split()) >= MIN_ABSTRACT_WORDS
        and len(record["keywords"]) >= MIN_KEYWORDS
    )


def main(email=MY_EMAIL, api_key=API_KEY, target=TARGET, pool=POOL, out=OUTPUT_PATH):
    base_params = {"tool": "keyword_project", "email": email}
    if api_key:
        base_params["api_key"] = api_key
    pause = 0.11 if api_key else 0.4  # NCBI limit: 3 requests/s, 10 with a key

    print("Searching PubMed ...")
    all_ids = search_ids(SEARCH_TERM, pool, base_params)
    print(f"Got {len(all_ids)} ids")

    kept = []
    batch_size = 200
    for start in range(0, len(all_ids), batch_size):
        batch = all_ids[start:start + batch_size]
        params = dict(base_params, db="pubmed", id=",".join(batch), retmode="xml")
        response = get_with_retries(EUTILS + "efetch.fcgi", params)

        for record in parse_articles(response.text):
            if is_good_record(record):
                kept.append(record)

        print(f"  fetched {start + len(batch)} / {len(all_ids)} ids, kept {len(kept)} papers")
        if len(kept) >= target:
            break
        time.sleep(pause)

    kept = kept[:target]
    if len(kept) < target:
        print(f"Warning: only {len(kept)} papers had an abstract and keywords. Try a bigger POOL value.")

    database = pd.DataFrame({
        "pmid": [r["pmid"] for r in kept],
        "title": [r["title"] for r in kept],
        "abstract": [r["abstract"] for r in kept],
        "keywords": [KEYWORD_SEPARATOR.join(k.replace(KEYWORD_SEPARATOR, ",") for k in r["keywords"])
                     for r in kept],
    })
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    database.to_csv(out, index=False)
    print(f"Saved {len(database)} documents to {out}")


main()

Install the project dependencies once from the repository root with `pip install -r requirements.txt`.

In [ ]:
"""Task 2: extract keywords with TF-IDF, RAKE, YAKE and TextRank and add them to the database."""
import re

import networkx as nx
import nltk
import pandas as pd
import yake
from rake_nltk import Rake
from sklearn.feature_extraction.text import TfidfVectorizer

for resource in ["punkt", "punkt_tab", "stopwords",
                 "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(resource, quiet=True)

from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords

DATABASE_PATH = "data/database.csv"
TOP_N = 20
RESULT_SIZE = 10
KEYWORD_SEPARATOR = ";"

english_stopwords = set(stopwords.words("english"))

# section labels such as "BACKGROUND:" or "METHODS:" are not content
SECTION_LABEL = re.compile(r"\b[A-Z][A-Z ,/&-]{2,}:\s*")


def clean_abstract(text):
    return SECTION_LABEL.sub("", text)


def join_keywords(keywords):
    return KEYWORD_SEPARATOR.join(k.replace(KEYWORD_SEPARATOR, ",") for k in keywords)


def tfidf_keywords(abstracts, top_n=TOP_N):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]{2,}\b",
    )
    matrix = vectorizer.fit_transform(abstracts)
    vocabulary = vectorizer.get_feature_names_out()

    all_keywords = []
    for row_number in range(matrix.shape[0]):
        row = matrix.getrow(row_number)
        best_first = row.data.argsort()[::-1][:top_n]
        all_keywords.append([vocabulary[row.indices[i]] for i in best_first])
    return all_keywords


def rake_keywords(text, top_n=TOP_N):
    rake = Rake(max_length=3)
    rake.extract_keywords_from_text(text)
    unique = list(dict.fromkeys(rake.get_ranked_phrases()))
    return unique[:top_n]


yake_extractor = yake.KeywordExtractor(lan="en", n=3, dedupLim=0.9, top=TOP_N)


def yake_keywords(text, top_n=TOP_N):
    found = yake_extractor.extract_keywords(text)
    found.sort(key=lambda pair: pair[1])  # lower score = better keyword in YAKE
    return [keyword for keyword, score in found[:top_n]]


def textrank_keywords(text, top_n=TOP_N, window=4):
    tokens = [t.lower() for t in word_tokenize(text)]
    tagged = pos_tag(tokens)

    def is_candidate(word, tag):
        is_noun_or_adjective = tag.startswith("NN") or tag.startswith("JJ")
        return is_noun_or_adjective and word.isalpha() and len(word) > 2 and word not in english_stopwords

    flags = [is_candidate(word, tag) for word, tag in tagged]
    candidate_words = [word for (word, tag), ok in zip(tagged, flags) if ok]
    if len(candidate_words) < 2:
        return candidate_words[:top_n]

    graph = nx.Graph()
    graph.add_nodes_from(candidate_words)
    for i, word in enumerate(candidate_words):
        for other in candidate_words[i + 1:i + window]:
            if other != word:
                graph.add_edge(word, other)

    word_scores = nx.pagerank(graph, alpha=0.85)

    phrase_scores = {}
    current_run = []
    for (word, tag), ok in zip(tagged + [("", "")], flags + [False]):  # dummy item closes the last run
        if ok:
            current_run.append(word)
            continue
        if current_run:
            run = current_run[:3]
            score = sum(word_scores[w] for w in run)
            phrase = " ".join(run)
            phrase_scores[phrase] = max(score, phrase_scores.get(phrase, 0))
            current_run = []

    ranked = sorted(phrase_scores.items(), key=lambda pair: pair[1], reverse=True)
    return [phrase for phrase, score in ranked[:top_n]]


def main(path=DATABASE_PATH):
    database = pd.read_csv(path, dtype=str).fillna("")
    abstracts = [clean_abstract(text) for text in database["abstract"]]
    print(f"{len(abstracts)} documents loaded")

    all_results = {}

    print("TF-IDF ...")
    all_results["tfidf"] = tfidf_keywords(abstracts)

    one_by_one = {"rake": rake_keywords, "yake": yake_keywords, "textrank": textrank_keywords}
    for method_name, extract in one_by_one.items():
        print(f"{method_name} ...")
        all_results[method_name] = []
        for number, abstract in enumerate(abstracts, start=1):
            all_results[method_name].append(extract(abstract))
            if number % 100 == 0:
                print(f"  {number} / {len(abstracts)}")

    for method_name, keyword_lists in all_results.items():
        database[f"{method_name}_keywords"] = [join_keywords(k[:RESULT_SIZE]) for k in keyword_lists]
        database[f"{method_name}_top{TOP_N}"] = [join_keywords(k) for k in keyword_lists]

    database.to_csv(path, index=False)
    print(f"Saved results in {path}")

    print("\nTitle          :", database.loc[0, "title"])
    print("Author keywords:", database.loc[0, "keywords"])
    for method_name in all_results:
        print(f"{method_name:<15}:", database.loc[0, f"{method_name}_keywords"])


main()

In [ ]:
"""Task 3: compare the extracted keywords with the author keywords (precision, recall, F1, PR-AUC)."""
import os
import re
from functools import lru_cache

import matplotlib.pyplot as plt
import pandas as pd
from nltk.stem import PorterStemmer
from sklearn.metrics import auc

DATABASE_PATH = "data/database.csv"
RESULTS_FOLDER = "results"
METHOD_COLUMNS = ["tfidf_keywords", "rake_keywords", "yake_keywords", "textrank_keywords"]
TAG = "task3"
MAX_K = 10
KEYWORD_SEPARATOR = ";"

stemmer = PorterStemmer()


def split_keywords(cell):
    return [word.strip() for word in cell.split(KEYWORD_SEPARATOR) if word.strip()]


@lru_cache(maxsize=None)
def stem_word(word):
    return stemmer.stem(word)


def normalize_phrase(phrase):
    words = re.findall(r"[a-z0-9]+", phrase.lower())
    return " ".join(stem_word(word) for word in words)


def precision_recall_at_k(predicted, reference, k):
    reference_set = {normalize_phrase(r) for r in reference}
    reference_set.discard("")

    unique_predicted = []
    for phrase in predicted:
        normalized = normalize_phrase(phrase)
        if normalized and normalized not in unique_predicted:
            unique_predicted.append(normalized)

    top_k = unique_predicted[:k]
    if not top_k or not reference_set:
        return 0.0, 0.0

    hits = sum(1 for phrase in top_k if phrase in reference_set)
    return hits / len(top_k), hits / len(reference_set)


def f1_score(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def evaluate_method(predictions, references, max_k=MAX_K):
    precision_curve, recall_curve = [], []
    for k in range(1, max_k + 1):
        scores = [precision_recall_at_k(p, r, k) for p, r in zip(predictions, references)]
        precision_curve.append(sum(s[0] for s in scores) / len(scores))
        recall_curve.append(sum(s[1] for s in scores) / len(scores))

    per_document = [precision_recall_at_k(p, r, max_k) for p, r in zip(predictions, references)]
    f1_values = [f1_score(p, r) for p, r in per_document]

    r_precision_values = []
    for predicted, reference in zip(predictions, references):
        cutoff = max(len({normalize_phrase(r) for r in reference} - {""}), 1)
        r_precision_values.append(precision_recall_at_k(predicted, reference, cutoff)[0])

    # recall starts at 0, so the curve is anchored there with the precision at k=1
    pr_auc = auc([0.0] + recall_curve, [precision_curve[0]] + precision_curve)

    return {
        "precision": precision_curve[-1],
        "recall": recall_curve[-1],
        "f1": sum(f1_values) / len(f1_values),
        "r_precision": sum(r_precision_values) / len(r_precision_values),
        "pr_auc": pr_auc,
        "precision_curve": precision_curve,
        "recall_curve": recall_curve,
    }


def author_keywords_in_abstract(abstracts, references):
    found, total = 0, 0
    for text, reference in zip(abstracts, references):
        text_normalized = " " + normalize_phrase(text) + " "
        for keyword in reference:
            keyword_normalized = normalize_phrase(keyword)
            if not keyword_normalized:
                continue
            total += 1
            if " " + keyword_normalized + " " in text_normalized:
                found += 1
    return found / total if total else 0.0


def make_table(results):
    rows = []
    for name, result in results.items():
        rows.append({
            "method": name,
            f"P@{MAX_K}": round(result["precision"], 4),
            f"R@{MAX_K}": round(result["recall"], 4),
            f"F1@{MAX_K}": round(result["f1"], 4),
            "R-precision": round(result["r_precision"], 4),
            "PR-AUC": round(result["pr_auc"], 4),
        })
    return pd.DataFrame(rows)


def plot_pr_curves(results, title, save_path):
    plt.figure(figsize=(7, 5))
    for name, result in results.items():
        plt.plot(result["recall_curve"], result["precision_curve"], marker="o", markersize=3,
                 label=f"{name} (AUC = {result['pr_auc']:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()


def main(path=DATABASE_PATH, columns=METHOD_COLUMNS, tag=TAG):
    database = pd.read_csv(path, dtype=str).fillna("")
    references = [split_keywords(cell) for cell in database["keywords"]]

    results = {}
    for column in columns:
        predictions = [split_keywords(cell) for cell in database[column]]
        results[column.replace("_keywords", "")] = evaluate_method(predictions, references)

    table = make_table(results)
    print(table.to_string(index=False))

    ceiling = author_keywords_in_abstract(database["abstract"].tolist(), references)
    average_length = sum(len(r) for r in references) / len(references)
    print(f"\nAuthor keywords that appear in the abstract: {ceiling:.1%}")
    print(f"Average number of author keywords per paper: {average_length:.2f}")

    os.makedirs(RESULTS_FOLDER, exist_ok=True)
    table.to_csv(f"{RESULTS_FOLDER}/metrics_{tag}.csv", index=False)
    plot_pr_curves(results, f"Precision-recall curves ({tag})", f"{RESULTS_FOLDER}/pr_curves_{tag}.png")
    return results


if __name__ == "__main__":
    main()

The fuzzy matching dependency is included in `requirements.txt`; install it there rather than running a shell command as Python.

In [ ]:
"""Task 4: take the top-20 candidates of each method, keep the 10 that best match the title (FuzzyWuzzy),
then evaluate them like in task 3.
 
Needs the functions of the task 3 cell (split_keywords, evaluate_method, make_table, plot_pr_curves),
so run the task 3 cell first (same notebook session).
"""
import re
 
import pandas as pd
from fuzzywuzzy import fuzz
from nltk.corpus import stopwords
 
DATABASE_PATH = "data/database.csv"
RESULTS_FOLDER = "results"
METHODS = ["tfidf", "rake", "yake", "textrank"]
TAG = "task4"
KEEP = 10
 
stop_set = set(stopwords.words("english"))
 
 
def content_tokens(text):
    # stopwords would match the title ("of", "and") and say nothing about the topic
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in stop_set and len(w) > 1]
 
 
def title_similarity(candidate, title_tokens):
    # best fuzzy match of every candidate word against the title words, then averaged over the candidate
    words = content_tokens(candidate)
    if not words or not title_tokens:
        return 0.0
    best_per_word = [max(fuzz.ratio(w, t) for t in title_tokens) for w in words]
    return sum(best_per_word) / len(best_per_word)
 
 
def filter_by_title(candidates, title, keep=KEEP):
    title_tokens = content_tokens(title)
    scored = [(title_similarity(c, title_tokens), position, c) for position, c in enumerate(candidates)]
    # highest similarity first; ties keep the order the algorithm gave (smaller position first)
    scored.sort(key=lambda item: (-item[0], item[1]))
    return [c for score, position, c in scored[:keep]]
 
 
def main(path=DATABASE_PATH):
    database = pd.read_csv(path, dtype=str).fillna("")
    references = [split_keywords(cell) for cell in database["keywords"]]
 
    results, baseline = {}, {}
    for method in METHODS:
        top20 = [split_keywords(cell) for cell in database[f"{method}_top20"]]
        filtered = [filter_by_title(c, t) for c, t in zip(top20, database["title"])]
        database[f"{method}_title_fuzzy"] = [";".join(f) for f in filtered]
        results[method] = evaluate_method(filtered, references)
 
        # task 3 result (plain top-10) kept next to it to see what the title filter changed
        plain = [split_keywords(cell) for cell in database[f"{method}_keywords"]]
        baseline[method] = evaluate_method(plain, references)
 
    table = make_table(results)
    table["PR-AUC task3"] = [round(baseline[m]["pr_auc"], 4) for m in METHODS]
    table["F1 task3"] = [round(baseline[m]["f1"], 4) for m in METHODS]
    print(table.to_string(index=False))
 
    database.to_csv(path, index=False)
    table.to_csv(f"{RESULTS_FOLDER}/metrics_{TAG}.csv", index=False)
    plot_pr_curves(results, f"Precision-recall curves ({TAG}, fuzzy title filter)",
                   f"{RESULTS_FOLDER}/pr_curves_{TAG}.png")
 
    print("\nTitle   :", database.loc[0, "title"])
    for method in METHODS:
        print(f"{method:<9}:", database.loc[0, f"{method}_title_fuzzy"])
    return results
 
 
main()

In [ ]:
import nltk
nltk.download("wordnet")

In [ ]:
"""Task 5: same filter as task 4, but the match with the title uses WordNet semantic similarity (Wu-Palmer).

Run the task 4 cell first: this cell reuses its functions and settings (stem_word, split_keywords,
evaluate_method, make_table, plot_pr_curves, content_tokens, METHODS, KEEP, RESULTS_FOLDER).
"""
from functools import lru_cache

import nltk
import pandas as pd
from nltk.corpus import wordnet

nltk.download("wordnet", quiet=True)

DATABASE_PATH = "data/database.csv"
TAG5 = "task5"
SENSES_PER_WORD = 3  # only the 3 most common meanings of a word, otherwise it is very slow


@lru_cache(maxsize=None)
def word_similarity(first_word, second_word):
    if first_word == second_word or stem_word(first_word) == stem_word(second_word):
        return 1.0
    best = 0.0
    for first in wordnet.synsets(first_word)[:SENSES_PER_WORD]:
        for second in wordnet.synsets(second_word)[:SENSES_PER_WORD]:
            score = first.wup_similarity(second)
            if score and score > best:
                best = score
    return best


def semantic_score(candidate, title_tokens):
    words = content_tokens(candidate)
    if not words or not title_tokens:
        return 0.0
    best_per_word = [max(word_similarity(w, t) for t in title_tokens) for w in words]
    return sum(best_per_word) / len(best_per_word)


def filter_by_semantics(candidates, title, keep=KEEP):
    title_tokens = content_tokens(title)
    scored = [(semantic_score(c, title_tokens), position, c) for position, c in enumerate(candidates)]
    scored.sort(key=lambda item: (-item[0], item[1]))
    return [c for score, position, c in scored[:keep]]


def main(path=DATABASE_PATH):
    database = pd.read_csv(path, dtype=str).fillna("")
    references = [split_keywords(cell) for cell in database["keywords"]]

    results, task3_results, task4_results = {}, {}, {}
    for method in METHODS:
        print(f"{method} ...")
        top20 = [split_keywords(cell) for cell in database[f"{method}_top20"]]
        filtered = [filter_by_semantics(c, t) for c, t in zip(top20, database["title"])]
        database[f"{method}_title_wordnet"] = [";".join(f) for f in filtered]
        results[method] = evaluate_method(filtered, references)

        for store, column in [(task3_results, f"{method}_keywords"), (task4_results, f"{method}_title_fuzzy")]:
            store[method] = evaluate_method([split_keywords(cell) for cell in database[column]], references)

    table = make_table(results)
    table["F1 task3"] = [round(task3_results[m]["f1"], 4) for m in METHODS]
    table["F1 task4"] = [round(task4_results[m]["f1"], 4) for m in METHODS]
    table["PR-AUC task3"] = [round(task3_results[m]["pr_auc"], 4) for m in METHODS]
    table["PR-AUC task4"] = [round(task4_results[m]["pr_auc"], 4) for m in METHODS]
    print(table.to_string(index=False))

    database.to_csv(path, index=False)
    table.to_csv(f"{RESULTS_FOLDER}/metrics_{TAG5}.csv", index=False)
    plot_pr_curves(results, f"Precision-recall curves ({TAG5}, WordNet title filter)",
                   f"{RESULTS_FOLDER}/pr_curves_{TAG5}.png")

    print("\nTitle   :", database.loc[0, "title"])
    for method in METHODS:
        print(f"{method:<9}:", database.loc[0, f"{method}_title_wordnet"])
    return results


if __name__ == "__main__":
    main()

In [ ]:
import requests

ITEM = "pubmed2018_w2v_200D.tar"

info = requests.get(f"https://archive.org/metadata/{ITEM}", timeout=60).json()
for file_info in info["files"]:
    print(file_info["name"], file_info.get("size"))

In [ ]:
import os
import tarfile
from urllib.parse import quote

import requests

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

candidates = [f for f in info["files"]
              if "200D" in f["name"] and f["name"].lower().endswith((".bin", ".tar", ".tgz", ".tar.gz"))]
candidates.sort(key=lambda f: not f["name"].endswith(".bin"))
chosen = candidates[0]
print("Downloading:", chosen["name"], chosen.get("size"))

url = f"https://archive.org/download/{ITEM}/{quote(chosen['name'])}"
target = os.path.join(MODEL_DIR, os.path.basename(chosen["name"]))
with requests.get(url, stream=True, timeout=60) as response:
    response.raise_for_status()
    with open(target, "wb") as out:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            out.write(chunk)

if target.endswith((".tar", ".tgz", ".tar.gz")):
    with tarfile.open(target) as archive:
        for member in archive.getmembers():
            if member.name.endswith("pubmed2018_w2v_200D.bin"):
                member.name = os.path.basename(member.name)
                archive.extract(member, MODEL_DIR)

print(os.listdir(MODEL_DIR))

In [ ]:
"""Task 6: same filter as task 4 and 5, but the match with the title uses word2vec cosine similarity.

Run the task 4 and task 5 cells first (this cell reuses stem_word, split_keywords, evaluate_method,
make_table, plot_pr_curves, content_tokens, METHODS, KEEP, RESULTS_FOLDER).
Needs the file pubmed2018_w2v_200D.bin saved at MODEL_PATH. No gensim needed: the binary file is read directly.
"""
import mmap
import os
from functools import lru_cache

import numpy as np
import pandas as pd

DATABASE_PATH = "data/database.csv"
MODEL_PATH = "models/pubmed2018_w2v_200D.bin"
TAG6 = "task6"

vectors = None  # filled in main()


class WordVectors:
    def __init__(self, table):
        self.table = table  # word -> unit-length vector

    def __contains__(self, word):
        return word in self.table

    def similarity(self, first_word, second_word):
        return float(np.dot(self.table[first_word], self.table[second_word]))


def load_needed_vectors(path, needed_words):
    # reads the word2vec binary file once and keeps only the words we need, so memory stays small
    if not os.path.exists(path):
        raise FileNotFoundError(f"word2vec file not found: {path}")
    table = {}
    with open(path, "rb") as handle:
        data = mmap.mmap(handle.fileno(), 0, access=mmap.ACCESS_READ)
        header_end = data.find(b"\n")
        vocab_size, dimension = (int(x) for x in data[:header_end].split())
        step = dimension * 4  # every number is a 4-byte float
        position = header_end + 1
        for number in range(vocab_size):
            word_end = data.find(b" ", position)
            word = data[position:word_end].decode("utf-8", errors="ignore")
            position = word_end + 1
            if word in needed_words:
                vector = np.frombuffer(data[position:position + step], dtype=np.float32)
                length = np.linalg.norm(vector)
                if length > 0:
                    table[word] = vector / length
            position += step
            if data[position:position + 1] == b"\n":  # some files put a newline after every vector
                position += 1
            if (number + 1) % 500000 == 0:
                print(f"  read {number + 1} / {vocab_size} words")
        data.close()
    return WordVectors(table)


def collect_words(database):
    words = set()
    for column in ["title"] + [f"{m}_top20" for m in METHODS]:
        for cell in database[column]:
            words.update(content_tokens(cell))
    return words


@lru_cache(maxsize=None)
def w2v_similarity(first_word, second_word):
    if first_word == second_word or stem_word(first_word) == stem_word(second_word):
        return 1.0
    if first_word not in vectors or second_word not in vectors:
        return 0.0
    # negative cosine means "unrelated", so it is treated like 0
    return max(0.0, vectors.similarity(first_word, second_word))


def w2v_score(candidate, title_tokens):
    words = content_tokens(candidate)
    if not words or not title_tokens:
        return 0.0
    best_per_word = [max(w2v_similarity(w, t) for t in title_tokens) for w in words]
    return sum(best_per_word) / len(best_per_word)


def filter_by_w2v(candidates, title, keep=KEEP):
    title_tokens = content_tokens(title)
    scored = [(w2v_score(c, title_tokens), position, c) for position, c in enumerate(candidates)]
    scored.sort(key=lambda item: (-item[0], item[1]))
    return [c for score, position, c in scored[:keep]]


def main(path=DATABASE_PATH, model_path=MODEL_PATH):
    global vectors
    database = pd.read_csv(path, dtype=str).fillna("")
    references = [split_keywords(cell) for cell in database["keywords"]]

    needed = collect_words(database)
    print(f"Loading word2vec vectors for {len(needed)} distinct words (takes a few minutes) ...")
    vectors = load_needed_vectors(model_path, needed)
    covered = sum(1 for w in needed if w in vectors)
    print(f"Words found in the word2vec vocabulary: {covered / len(needed):.1%}")

    results = {}
    earlier = {"task3": {}, "task4": {}, "task5": {}}
    earlier_columns = {"task3": "keywords", "task4": "title_fuzzy", "task5": "title_wordnet"}

    for method in METHODS:
        print(f"{method} ...")
        top20 = [split_keywords(cell) for cell in database[f"{method}_top20"]]
        filtered = [filter_by_w2v(c, t) for c, t in zip(top20, database["title"])]
        database[f"{method}_title_w2v"] = [";".join(f) for f in filtered]
        results[method] = evaluate_method(filtered, references)

        for name, suffix in earlier_columns.items():
            column = f"{method}_{suffix}"
            earlier[name][method] = evaluate_method([split_keywords(cell) for cell in database[column]], references)

    table = make_table(results)
    for name in earlier:
        table[f"F1 {name}"] = [round(earlier[name][m]["f1"], 4) for m in METHODS]
    for name in earlier:
        table[f"PR-AUC {name}"] = [round(earlier[name][m]["pr_auc"], 4) for m in METHODS]
    print(table.to_string(index=False))

    database.to_csv(path, index=False)
    table.to_csv(f"{RESULTS_FOLDER}/metrics_{TAG6}.csv", index=False)
    plot_pr_curves(results, f"Precision-recall curves ({TAG6}, word2vec title filter)",
                   f"{RESULTS_FOLDER}/pr_curves_{TAG6}.png")

    print("\nTitle   :", database.loc[0, "title"])
    for method in METHODS:
        print(f"{method:<9}:", database.loc[0, f"{method}_title_w2v"])
    return results


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
from nltk.corpus import wordnet

database = pd.read_csv("data/database.csv", dtype=str).fillna("")


def average_overlap(first_column, second_column):
    overlaps = []
    for a, b in zip(database[first_column], database[second_column]):
        set_a, set_b = set(split_keywords(a)), set(split_keywords(b))
        overlaps.append(len(set_a & set_b) / max(len(set_a | set_b), 1))
    return sum(overlaps) / len(overlaps)


print("Average overlap of the selected 10 keywords (100% = identical sets)")
for method in METHODS:
    t4_t5 = average_overlap(f"{method}_title_fuzzy", f"{method}_title_wordnet")
    t4_t6 = average_overlap(f"{method}_title_fuzzy", f"{method}_title_w2v")
    t5_t6 = average_overlap(f"{method}_title_wordnet", f"{method}_title_w2v")
    print(f"{method:<9} task4-task5: {t4_t5:.1%}   task4-task6: {t4_t6:.1%}   task5-task6: {t5_t6:.1%}")

words = collect_words(database)
in_wordnet = sum(1 for w in words if wordnet.synsets(w)) / len(words)
in_word2vec = sum(1 for w in words if w in vectors) / len(words)
print(f"\nDistinct words checked: {len(words)}")
print(f"Found in WordNet : {in_wordnet:.1%}")
print(f"Found in word2vec: {in_word2vec:.1%}")

In [ ]:
"""Task 7: supervised keyword ranking.

Every 1-3 word phrase of title + abstract is a candidate. A classifier learns from the author keywords of the
training documents which candidates are keywords, then the 10 best-scored candidates of each test document
are the prediction (20% of the documents are kept for testing).

Run the task 4 cell first: this cell reuses stop_set, stem_word, normalize_phrase, split_keywords,
evaluate_method, make_table, plot_pr_curves, METHODS, KEEP, RESULTS_FOLDER and KEYWORD_SEPARATOR.
"""
import math
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATABASE_PATH = "data/database.csv"
TAG7 = "task7"
MAX_WORDS = 3
TEST_SIZE = 0.2
SEED = 42
FEATURES = (["n_words", "n_chars", "has_digit", "tf", "first_position", "in_title", "title_overlap",
             "mean_idf", "max_idf", "min_idf", "tf_idf", "log_phrase_df"]
            + [f"{m}_score" for m in METHODS] + ["n_methods"])


def extract_candidates(title, abstract):
    text = f"{title}. {abstract}".lower()
    candidates, offset = {}, 0
    for chunk in re.split(r"[.,;:()\[\]{}?!\"“”‘’/]", text):  # a phrase never crosses punctuation
        words = re.findall(r"[a-z0-9]+", chunk)
        for start in range(len(words)):
            for size in range(1, MAX_WORDS + 1):
                phrase = words[start:start + size]
                if len(phrase) < size:
                    break
                if (phrase[0] in stop_set or phrase[-1] in stop_set or min(len(w) for w in phrase) < 2
                        or all(w.isdigit() for w in phrase)):
                    continue
                key = " ".join(stem_word(w) for w in phrase)
                if key in candidates:
                    candidates[key]["count"] += 1
                else:
                    candidates[key] = {"surface": " ".join(phrase), "count": 1, "first": offset + start, "words": size}
        offset += len(words)
    return candidates, offset


def method_scores(cell):
    # the position a method gave to a phrase, turned into a score: first place = 1.0, 20th place = 0.05
    scores = {}
    for position, phrase in enumerate(split_keywords(cell)[:20]):
        key = normalize_phrase(phrase)
        if key and key not in scores:
            scores[key] = (20 - position) / 20
    return scores


def build_table(database):
    per_document = [extract_candidates(t, a) for t, a in zip(database["title"], database["abstract"])]
    word_frequency, phrase_frequency = Counter(), Counter()
    for candidates, length in per_document:
        phrase_frequency.update(candidates.keys())
        word_frequency.update({w for key in candidates for w in key.split()})
    n_docs = len(database)

    rows = []
    for doc, (candidates, length) in enumerate(per_document):
        title_stems = [stem_word(w) for w in re.findall(r"[a-z0-9]+", database["title"][doc].lower())]
        title_text, title_set = " " + " ".join(title_stems) + " ", set(title_stems)
        reference_keys = {normalize_phrase(r) for r in split_keywords(database["keywords"][doc])} - {""}
        scores = {m: method_scores(database[f"{m}_top20"][doc]) for m in METHODS}

        for key, info in candidates.items():
            words = key.split()
            idfs = [math.log((n_docs + 1) / (word_frequency[w] + 1)) + 1 for w in words]
            row = {
                "doc": doc, "phrase": info["surface"], "key": key,
                "n_words": info["words"], "n_chars": len(key),
                "has_digit": int(any(ch.isdigit() for ch in key)),
                "tf": info["count"], "first_position": info["first"] / max(length, 1),
                "in_title": int(f" {key} " in title_text),
                "title_overlap": sum(w in title_set for w in words) / len(words),
                "mean_idf": sum(idfs) / len(idfs), "max_idf": max(idfs), "min_idf": min(idfs),
                "tf_idf": math.log1p(info["count"]) * sum(idfs) / len(idfs),
                "log_phrase_df": math.log(phrase_frequency[key]),
                "label": int(key in reference_keys),
            }
            for m in METHODS:
                row[f"{m}_score"] = scores[m].get(key, 0.0)
            row["n_methods"] = sum(row[f"{m}_score"] > 0 for m in METHODS)
            rows.append(row)
    return pd.DataFrame(rows)


def best_phrases(part, probabilities, docs, keep=KEEP):
    ordered = part.assign(score=probabilities).sort_values(["doc", "score"], ascending=[True, False])
    top = {doc: group["phrase"].head(keep).tolist() for doc, group in ordered.groupby("doc")}
    return [top.get(doc, []) for doc in docs]


def main(path=DATABASE_PATH):
    database = pd.read_csv(path, dtype=str).fillna("")
    references = [split_keywords(cell) for cell in database["keywords"]]

    table = build_table(database)
    train_docs, test_docs = train_test_split(np.arange(len(database)), test_size=TEST_SIZE, random_state=SEED)
    test_docs = np.sort(test_docs)
    train = table[table["doc"].isin(train_docs)]
    test = table[table["doc"].isin(test_docs)]
    print(f"Candidates: {len(table)} rows, {table['label'].mean():.2%} of them are author keywords")
    print(f"Training documents: {len(train_docs)}, test documents: {len(test_docs)}")

    keys_by_doc = table.groupby("doc")["key"].agg(set)
    coverage = []
    for doc in test_docs:
        reference_keys = {normalize_phrase(r) for r in references[doc]} - {""}
        coverage.append(len(reference_keys & keys_by_doc.get(doc, set())) / max(len(reference_keys), 1))
    print(f"Author keywords present among the candidates (upper bound of recall): {np.mean(coverage):.1%}")

    models = {
        "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
        "gradient boosting": HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, l2_regularization=1.0,
                                                           early_stopping=False, random_state=SEED),
    }

    test_references = [references[d] for d in test_docs]
    predictions = {}
    for method in METHODS:  # the unsupervised methods of task 3, on the same test documents
        predictions[method] = [split_keywords(database[f"{method}_keywords"][d]) for d in test_docs]
    if "yake_title_w2v" in database.columns:
        predictions["yake + title w2v"] = [split_keywords(database["yake_title_w2v"][d]) for d in test_docs]

    for name, model in models.items():
        print(f"Training {name} ...")
        model.fit(train[FEATURES], train["label"])
        predictions[name] = best_phrases(test, model.predict_proba(test[FEATURES])[:, 1], test_docs)

    results = {name: evaluate_method(pred, test_references) for name, pred in predictions.items()}
    metrics = make_table(results)
    print(metrics.to_string(index=False))

    forest = models["gradient boosting"]
    importance = permutation_importance(forest, test[FEATURES], test["label"], scoring="average_precision",
                                        n_repeats=3, random_state=SEED)
    ranking = pd.Series(importance.importances_mean, index=FEATURES).sort_values(ascending=False)
    print("\nFeature importance (gradient boosting, drop in average precision when the feature is shuffled):")
    print(ranking.round(4).to_string())

    coefficients = pd.Series(models["logistic regression"][-1].coef_[0], index=FEATURES).sort_values(ascending=False)
    print("\nLogistic regression coefficients (standardized features):")
    print(coefficients.round(3).to_string())

    metrics.to_csv(f"{RESULTS_FOLDER}/metrics_{TAG7}.csv", index=False)
    plot_pr_curves(results, f"Precision-recall curves ({TAG7}, supervised, test 20%)",
                   f"{RESULTS_FOLDER}/pr_curves_{TAG7}.png")
    saved = pd.DataFrame({"pmid": database["pmid"][test_docs].values, "title": database["title"][test_docs].values,
                          "author_keywords": database["keywords"][test_docs].values})
    for name in models:
        saved[name] = [KEYWORD_SEPARATOR.join(p) for p in predictions[name]]
    saved.to_csv(f"{RESULTS_FOLDER}/{TAG7}_test_predictions.csv", index=False)

    print("\nTitle   :", saved.loc[0, "title"])
    print("Authors :", saved.loc[0, "author_keywords"])
    for name in models:
        print(f"{name:<20}:", saved.loc[0, name])
    return results


if __name__ == "__main__":
    main()

In [ ]:
"""Task 8, step 1: write the database as json-lines files that SEKE's preprocessing.py can read.

Same 80/20 split as task 7 (test_size=0.2, random_state=42), so SEKE is tested on the same 200 documents.
Only needs pandas and scikit-learn: run it in the normal notebook, not in the seke environment.
"""
import json
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATABASE_PATH = "data/database.csv"
SEKE_FOLDER = os.environ.get("SEKE_FOLDER", "SEKE_keyword_extraction")
OUTPUT_FOLDER = os.path.join(SEKE_FOLDER, "data", "pubmed")
TEST_SIZE = 0.2
SEED = 42


def write_json_lines(frame, path):
    with open(path, "w", encoding="utf8") as out:
        for _, row in frame.iterrows():
            record = {"pmid": row["pmid"], "title": row["title"], "abstract": row["abstract"],
                      "keywords": row["keywords"]}
            out.write(json.dumps(record, ensure_ascii=False) + "\n")


def main():
    database = pd.read_csv(DATABASE_PATH, dtype=str).fillna("")
    train_docs, test_docs = train_test_split(np.arange(len(database)), test_size=TEST_SIZE, random_state=SEED)
    train_docs, test_docs = np.sort(train_docs), np.sort(test_docs)

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    write_json_lines(database.iloc[train_docs], os.path.join(OUTPUT_FOLDER, "train_all.jsonl"))
    write_json_lines(database.iloc[test_docs], os.path.join(OUTPUT_FOLDER, "test.jsonl"))
    print(f"Written to {OUTPUT_FOLDER}: {len(train_docs)} training and {len(test_docs)} test documents")
    print("First test document:", database.loc[test_docs[0], "title"])


if __name__ == "__main__":
    main()

In [ ]:
"""Task 8, step 3: evaluate the SEKE outputs with the same metrics as all the other tasks.

Run the task 4 cell first (this cell reuses split_keywords, normalize_phrase, evaluate_method, make_table,
plot_pr_curves, METHODS, RESULTS_FOLDER). SEKE prints its own scores in the console, but those only count author
keywords that appear in the text; here every author keyword counts, like in tasks 3 to 7.
Every run listed in SEKE_RUNS that exists on disk is evaluated; missing files are skipped.
"""
import os
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

required = ["split_keywords", "evaluate_method", "make_table", "plot_pr_curves", "METHODS", "RESULTS_FOLDER"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"Run the task 4 cell first. Missing: {missing}")

DATABASE_PATH = "data/database.csv"
SEKE_RESULTS_FOLDER = os.environ.get("SEKE_RESULTS_FOLDER", os.path.join(os.environ.get("SEKE_FOLDER", "SEKE_keyword_extraction"), "results"))
SEKE_RUNS = {
    "SEKE (default config)": os.path.join(SEKE_RESULTS_FOLDER, "pubmed_moe_rnn_default_seed_42.tsv"),
}
TASK7_PREDICTIONS = f"{RESULTS_FOLDER}/task7_test_predictions.csv"
TAG8 = "task8"
TEST_SIZE = 0.2
SEED = 42


def read_seke_run(path, database, test_docs):
    seke = pd.read_csv(path, sep="\t", dtype=str).fillna("")
    if len(seke) != len(test_docs):
        raise ValueError(f"{path} has {len(seke)} rows, the test set has {len(test_docs)} documents")
    matches = 0
    for text, doc in zip(seke["text"], test_docs):
        title_words = re.findall(r"[a-z0-9]+", database["title"][doc].lower())[:3]
        text_words = re.findall(r"[a-z0-9]+", text.lower())[:3]
        matches += title_words == text_words
    print(f"Order check: {matches} of {len(test_docs)} rows start with the title of the expected test document")
    return seke


def main():
    database = pd.read_csv(DATABASE_PATH, dtype=str).fillna("")
    _, test_docs = train_test_split(np.arange(len(database)), test_size=TEST_SIZE, random_state=SEED)
    test_docs = np.sort(test_docs)
    references = [split_keywords(database["keywords"][d]) for d in test_docs]

    predictions = {}
    for method in METHODS:
        predictions[method] = [split_keywords(database[f"{method}_keywords"][d]) for d in test_docs]
    if "yake_title_w2v" in database.columns:
        predictions["yake + title w2v"] = [split_keywords(database["yake_title_w2v"][d]) for d in test_docs]
    if os.path.exists(TASK7_PREDICTIONS):
        task7 = pd.read_csv(TASK7_PREDICTIONS, dtype=str).fillna("")
        if task7["pmid"].tolist() == database["pmid"][test_docs].tolist():
            for name in ["logistic regression", "gradient boosting"]:
                predictions[name] = [split_keywords(cell) for cell in task7[name]]
        else:
            print("Task 7 predictions belong to another test set, so they are left out")

    seke_tables = {}
    for name, path in SEKE_RUNS.items():
        if not os.path.exists(path):
            print(f"{name}: file not found, skipped ({path})")
            continue
        print(name)
        seke_tables[name] = read_seke_run(path, database, test_docs)
        predictions[name] = [split_keywords(cell) for cell in seke_tables[name]["preds"]]

    results = {name: evaluate_method(pred, references) for name, pred in predictions.items()}
    table = make_table(results)
    table["keywords per doc"] = [round(float(np.mean([len(p) for p in predictions[name]])), 2) for name in results]
    print(table.to_string(index=False))

    table.to_csv(f"{RESULTS_FOLDER}/metrics_{TAG8}.csv", index=False)
    plot_pr_curves(results, f"Precision-recall curves ({TAG8}, SEKE, test 20%)",
                   f"{RESULTS_FOLDER}/pr_curves_{TAG8}.png")

    for name, seke in seke_tables.items():
        print(f"\n{name}: documents without any keyword:", sum(1 for p in predictions[name] if not p))
    if seke_tables:
        name, seke = next(iter(seke_tables.items()))
        first = test_docs[0]
        print("\nTitle   :", database["title"][first])
        print("Authors :", database["keywords"][first])
        print(f"{name}:", seke.loc[0, "preds"])
        print("Scores  :", seke.loc[0, "scores"])
    return results


if __name__ == "__main__":
    main()

In [ ]:
"""Task 9: the supervised ranker of task 7, extended with word2vec embedding features and a diversity re-ranking.

Embedding features follow the idea of EmbedRank (Bennani-Smires et al., 2018): a good keyphrase is close to the
whole document in embedding space. The re-ranking is a word-overlap version of the diversity idea (MMR) of the same
paper: a candidate loses part of its score when it shares words with a keyword that is already chosen. The size of
that penalty is chosen by 5-fold cross-validation on the training papers only (penalty 0 means "no diversity").
Three variants are compared on the same 200 test documents as task 7:
  1. the features of task 7,  2. plus embedding features,  3. plus embedding features and diversity re-ranking.

Run first, in this order: the task 4, task 6 and task 7 cells (this cell reuses their functions and settings).
"""
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold, train_test_split

required = ["content_tokens", "evaluate_method", "make_table", "plot_pr_curves", "split_keywords", "METHODS", "KEEP",
            "RESULTS_FOLDER", "KEYWORD_SEPARATOR", "build_table", "FEATURES", "load_needed_vectors", "MODEL_PATH"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"Run the task 4, 6 and 7 cells first. Missing: {missing}")

DATABASE_PATH = "data/database.csv"
TAG9 = "task9"
TEST_SIZE = 0.2
SEED = 42
PENALTIES = [0.0, 0.1, 0.2, 0.3, 0.5, 1.0]  # candidate values of the diversity penalty
POOL = 40  # only the 40 most probable candidates of a paper take part in the re-ranking
EMBEDDING_FEATURES = ["sim_abstract", "sim_title", "known_fraction"]
ALL_FEATURES = FEATURES + EMBEDDING_FEATURES


def mean_vector(words, table):
    found = [table[w] for w in words if w in table]
    if not found:
        return None
    total = np.mean(found, axis=0)
    length = np.linalg.norm(total)
    return total / length if length > 0 else None


def add_embedding_features(table, database, embeddings):
    sim_abstract = np.zeros(len(table))
    sim_title = np.zeros(len(table))
    known = np.zeros(len(table))
    phrases = table["phrase"].to_numpy()
    cache = {}
    for doc, rows in table.groupby("doc").indices.items():
        title, abstract = database["title"][doc], database["abstract"][doc]
        doc_vector = mean_vector(content_tokens(f"{title} {abstract}"), embeddings.table)
        title_vector = mean_vector(content_tokens(title), embeddings.table)
        for row in rows:
            phrase = phrases[row]
            if phrase not in cache:
                words = phrase.split()
                cache[phrase] = (mean_vector(words, embeddings.table), sum(w in embeddings for w in words) / len(words))
            vector, fraction = cache[phrase]
            known[row] = fraction
            if vector is not None:
                if doc_vector is not None:
                    sim_abstract[row] = float(vector @ doc_vector)
                if title_vector is not None:
                    sim_title[row] = float(vector @ title_vector)
    table = table.copy()
    table["sim_abstract"], table["sim_title"], table["known_fraction"] = sim_abstract, sim_title, known
    return table


def rerank(group, penalty, keep=KEEP):
    # group: the candidates of one paper, best first; every step takes the candidate with the best
    # (probability - penalty * largest word overlap with the keywords already chosen)
    pool = group.head(POOL)
    phrases = pool["phrase"].tolist()
    words = [set(key.split()) for key in pool["key"]]
    scores = pool["score"].to_numpy()
    chosen, remaining = [], list(range(len(pool)))
    while remaining and len(chosen) < keep:
        best, best_value = None, -np.inf
        for i in remaining:
            redundancy = max((len(words[i] & words[j]) / len(words[i] | words[j]) for j in chosen), default=0.0)
            value = scores[i] - penalty * redundancy
            if value > best_value:
                best, best_value = i, value
        chosen.append(best)
        remaining.remove(best)
    return [phrases[i] for i in chosen]


def ranked_phrases(part, probabilities, docs, penalty=0.0):
    ordered = part.assign(score=probabilities).sort_values(["doc", "score"], ascending=[True, False])
    top = {doc: rerank(group, penalty) for doc, group in ordered.groupby("doc")}
    return [top.get(doc, []) for doc in docs]


def make_model():
    return HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, l2_regularization=1.0,
                                          early_stopping=False, random_state=SEED)


def choose_penalty(train, references):
    # out-of-fold probabilities for the training papers, then the penalty with the best F1@10 on them
    held_out = np.zeros(len(train))
    for fit_rows, test_rows in GroupKFold(n_splits=5).split(train, groups=train["doc"]):
        model = make_model().fit(train.iloc[fit_rows][ALL_FEATURES], train.iloc[fit_rows]["label"])
        held_out[test_rows] = model.predict_proba(train.iloc[test_rows][ALL_FEATURES])[:, 1]
    docs = np.sort(train["doc"].unique())
    docs_references = [references[d] for d in docs]
    best_penalty, best_f1, f1_by_penalty = 0.0, -1.0, {}
    for penalty in PENALTIES:
        f1 = evaluate_method(ranked_phrases(train, held_out, docs, penalty), docs_references)["f1"]
        f1_by_penalty[penalty] = f1
        if f1 > best_f1 + 1e-9:
            best_penalty, best_f1 = penalty, f1
    return best_penalty, f1_by_penalty


def main(path=DATABASE_PATH, model_path=MODEL_PATH):
    database = pd.read_csv(path, dtype=str).fillna("")
    references = [split_keywords(cell) for cell in database["keywords"]]

    table = build_table(database)
    needed = set()
    for title, abstract in zip(database["title"], database["abstract"]):
        needed.update(content_tokens(f"{title} {abstract}"))
    print(f"Loading word2vec vectors for {len(needed)} distinct words (takes a few minutes) ...")
    embeddings = load_needed_vectors(model_path, needed)
    print(f"Words found in the word2vec vocabulary: {sum(1 for w in needed if w in embeddings) / len(needed):.1%}")
    table = add_embedding_features(table, database, embeddings)

    train_docs, test_docs = train_test_split(np.arange(len(database)), test_size=TEST_SIZE, random_state=SEED)
    test_docs = np.sort(test_docs)
    train, test = table[table["doc"].isin(train_docs)], table[table["doc"].isin(test_docs)]
    test_references = [references[d] for d in test_docs]

    print("Training with the features of task 7 ...")
    basic = make_model().fit(train[FEATURES], train["label"])
    print("Training with the embedding features added ...")
    extended = make_model().fit(train[ALL_FEATURES], train["label"])
    print("Choosing the diversity penalty by 5-fold cross-validation on the training papers ...")
    penalty, f1_by_penalty = choose_penalty(train, references)
    print("F1@10 on the training papers (out-of-fold) for each penalty:")
    for value, f1 in f1_by_penalty.items():
        print(f"  penalty {value:.1f}: {f1:.4f}")
    print(f"Chosen penalty: {penalty}")

    basic_scores = basic.predict_proba(test[FEATURES])[:, 1]
    extended_scores = extended.predict_proba(test[ALL_FEATURES])[:, 1]
    names = ["task 7 features", "+ embedding features", f"+ embeddings + diversity (penalty {penalty})"]

    predictions = {}
    if "yake_title_w2v" in database.columns:
        predictions["yake + title w2v"] = [split_keywords(database["yake_title_w2v"][d]) for d in test_docs]
    predictions[names[0]] = ranked_phrases(test, basic_scores, test_docs)
    predictions[names[1]] = ranked_phrases(test, extended_scores, test_docs)
    predictions[names[2]] = ranked_phrases(test, extended_scores, test_docs, penalty)

    results = {name: evaluate_method(pred, test_references) for name, pred in predictions.items()}
    table_out = make_table(results)
    print("\n" + table_out.to_string(index=False))

    importance = permutation_importance(extended, test[ALL_FEATURES], test["label"], scoring="average_precision",
                                        n_repeats=3, random_state=SEED)
    ranking = pd.Series(importance.importances_mean, index=ALL_FEATURES).sort_values(ascending=False)
    print("\nFeature importance (drop in average precision when the feature is shuffled):")
    print(ranking.round(4).to_string())

    table_out.to_csv(f"{RESULTS_FOLDER}/metrics_{TAG9}.csv", index=False)
    plot_pr_curves(results, f"Precision-recall curves ({TAG9}, embeddings and diversity, test 20%)",
                   f"{RESULTS_FOLDER}/pr_curves_{TAG9}.png")
    saved = pd.DataFrame({"pmid": database["pmid"][test_docs].values, "title": database["title"][test_docs].values,
                          "author_keywords": database["keywords"][test_docs].values})
    for name in names:
        saved[name] = [KEYWORD_SEPARATOR.join(p) for p in predictions[name]]
    saved.to_csv(f"{RESULTS_FOLDER}/{TAG9}_test_predictions.csv", index=False)

    print("\nTitle   :", saved.loc[0, "title"])
    print("Authors :", saved.loc[0, "author_keywords"])
    for name in names:
        print(f"{name:<40}:", saved.loc[0, name])
    return results


if __name__ == "__main__":
    main()

In [ ]:
"""Task 10, step 1: read the three benchmark zip files, save them in the table format of the project and write
the json-lines files that SEKE needs (same 80/20 split idea as task 7: test_size=0.2, random_state=42).

Needs the task 4 cell (normalize_phrase). Runs in the normal notebook (environment mozafar).
The three zip files must be in the folder ZIP_FOLDER, next to the notebook, and must not be unzipped.
"""
import json
import os
import re
import zipfile

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

required = ["normalize_phrase"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"Run the task 4 cell first. Missing: {missing}")

ZIP_FOLDER = "benchmarks"
TABLE_FOLDER = "data/benchmarks"
SEKE_FOLDER = os.environ.get("SEKE_FOLDER", "SEKE_keyword_extraction")
DATASETS = ["Inspec", "kdd", "SemEval2017"]
TEST_SIZE = 0.2
SEED = 42


def clean_text(text):
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()


def read_dataset(name):
    path = os.path.join(ZIP_FOLDER, f"{name}.zip")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Put the file {name}.zip in the folder '{ZIP_FOLDER}' (next to the notebook)")
    rows = []
    with zipfile.ZipFile(path) as archive:
        names = set(archive.namelist())
        ids = sorted(n.split("/")[-1][:-4] for n in names if n.startswith(f"{name}/docsutf8/") and n.endswith(".txt"))
        for doc_id in ids:
            key_path = f"{name}/keys/{doc_id}.key"
            if key_path not in names:
                continue
            text = clean_text(archive.read(f"{name}/docsutf8/{doc_id}.txt").decode("utf8", errors="replace"))
            keys, seen = [], set()
            for line in archive.read(key_path).decode("utf8", errors="replace").splitlines():
                key = clean_text(line).replace(";", ",")
                if key and key.lower() not in seen:
                    seen.add(key.lower())
                    keys.append(key)
            if text and keys:
                rows.append({"pmid": doc_id, "title": "", "abstract": text, "keywords": ";".join(keys)})
    return pd.DataFrame(rows)


def describe(name, table):
    words = table["abstract"].str.split().map(len)
    gold = [{normalize_phrase(k) for k in cell.split(";")} - {""} for cell in table["keywords"]]
    present = sum(sum(f" {key} " in f" {normalize_phrase(text)} " for key in keys)
                  for text, keys in zip(table["abstract"], gold))
    total = sum(len(keys) for keys in gold)
    print(f"{name}: {len(table)} documents | words per document: mean {words.mean():.0f}, max {words.max()} | "
          f"gold keywords per document: mean {total / len(table):.2f}, min {min(map(len, gold))}, "
          f"max {max(map(len, gold))} | gold keywords found in the text: {present / total:.1%}")


def write_json_lines(frame, path):
    with open(path, "w", encoding="utf8") as out:
        for _, row in frame.iterrows():
            record = {"pmid": row["pmid"], "title": row["title"], "abstract": row["abstract"],
                      "keywords": row["keywords"]}
            out.write(json.dumps(record, ensure_ascii=False) + "\n")


def main():
    os.makedirs(TABLE_FOLDER, exist_ok=True)
    for name in DATASETS:
        table = read_dataset(name)
        describe(name, table)
        train_docs, test_docs = train_test_split(np.arange(len(table)), test_size=TEST_SIZE, random_state=SEED)
        table["split"] = "train"
        table.loc[np.sort(test_docs), "split"] = "test"
        table.to_csv(f"{TABLE_FOLDER}/{name}.csv", index=False)

        seke_folder = os.path.join(SEKE_FOLDER, "data", name)
        os.makedirs(seke_folder, exist_ok=True)
        train, test = table[table["split"] == "train"], table[table["split"] == "test"]
        write_json_lines(train, os.path.join(seke_folder, "train_all.jsonl"))
        write_json_lines(test, os.path.join(seke_folder, "test.jsonl"))
        print(f"   table saved in {TABLE_FOLDER}/{name}.csv; SEKE files written to {seke_folder}: "
              f"{len(train)} training and {len(test)} test documents; first test document id: {test['pmid'].iloc[0]}")


if __name__ == "__main__":
    main()

In [ ]:
"""Task 10, step 2: run the four extractors of task 2 (TF-IDF, RAKE, YAKE, TextRank) on the three benchmark tables
made in step 1 and add the columns <method>_keywords (top 10) and <method>_top20 to each table.

Run first: the task 2 cell (it defines the extractor functions) and the task 4 cell (METHODS), then step 1 of task 10.
The cell can be run again: the columns are simply overwritten.
"""
import pandas as pd

required = ["clean_abstract", "tfidf_keywords", "rake_keywords", "yake_keywords", "textrank_keywords",
            "join_keywords", "RESULT_SIZE", "TOP_N", "METHODS"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"Run the task 2 and task 4 cells first. Missing: {missing}")

TABLE_FOLDER = "data/benchmarks"
DATASETS = ["Inspec", "kdd", "SemEval2017"]


def main():
    for name in DATASETS:
        path = f"{TABLE_FOLDER}/{name}.csv"
        table = pd.read_csv(path, dtype=str).fillna("")
        abstracts = [clean_abstract(text) for text in table["abstract"]]
        print(f"\n===== {name}: {len(abstracts)} documents =====")

        all_results = {}
        print("tfidf ...")
        # same as task 2: the vocabulary is fitted on all documents of the dataset (no labels are used)
        all_results["tfidf"] = tfidf_keywords(abstracts)

        one_by_one = {"rake": rake_keywords, "yake": yake_keywords, "textrank": textrank_keywords}
        for method, extract in one_by_one.items():
            print(f"{method} ...")
            all_results[method] = []
            for number, abstract in enumerate(abstracts, start=1):
                all_results[method].append(extract(abstract))
                if number % 100 == 0:
                    print(f"  {number} / {len(abstracts)}")

        for method, keyword_lists in all_results.items():
            table[f"{method}_keywords"] = [join_keywords(k[:RESULT_SIZE]) for k in keyword_lists]
            table[f"{method}_top{TOP_N}"] = [join_keywords(k) for k in keyword_lists]

        table.to_csv(path, index=False)
        print(f"saved in {path}")
        for method in METHODS:
            print(f"  {method:<9}: {table.loc[0, f'{method}_keywords']}")


if __name__ == "__main__":
    main()

In [ ]:
"""Task 10, step 3: evaluate all algorithms on the three benchmark datasets (test part = 20% of each dataset).

Algorithms: the four extractors of task 2 (TF-IDF, RAKE, YAKE, TextRank), the supervised ranker of task 7
(logistic regression and gradient boosting, trained on the 80% part), the variant of task 9 (embedding features and
diversity re-ranking) and SEKE (task 8, trained per dataset). The datasets have no titles, so the title filters of
tasks 4 to 6 cannot be used and the title features of the supervised ranker are always zero.
The cell can be run again later: a missing SEKE file is skipped.

Run first: the task 4, task 6, task 7 and task 9 cells (this cell reuses their functions), then the cells of
steps 1 and 2 of task 10.
"""
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

required = ["split_keywords", "normalize_phrase", "evaluate_method", "make_table", "METHODS", "RESULTS_FOLDER",
            "KEYWORD_SEPARATOR", "content_tokens", "build_table", "FEATURES", "load_needed_vectors", "MODEL_PATH",
            "add_embedding_features", "choose_penalty", "ranked_phrases", "make_model", "ALL_FEATURES"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"Run the task 4, 6, 7 and 9 cells first. Missing: {missing}")

TABLE_FOLDER = "data/benchmarks"
DATASETS = ["Inspec", "kdd", "SemEval2017"]
SEKE_RESULTS_FOLDER = os.environ.get("SEKE_RESULTS_FOLDER", os.path.join(os.environ.get("SEKE_FOLDER", "SEKE_keyword_extraction"), "results"))
USE_TASK9_VARIANT = True
TAG10 = "task10"


def strict_scores(predictions, references, k=10):
    # precision divided by k even when fewer than k keywords were predicted (no advantage for short lists)
    precisions, f1_values = [], []
    for predicted, reference in zip(predictions, references):
        reference_set = {normalize_phrase(r) for r in reference} - {""}
        unique = []
        for phrase in predicted:
            normalized = normalize_phrase(phrase)
            if normalized and normalized not in unique:
                unique.append(normalized)
        hits = sum(1 for phrase in unique[:k] if phrase in reference_set)
        precision = hits / k
        recall = hits / len(reference_set) if reference_set else 0.0
        precisions.append(precision)
        f1_values.append(2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0)
    return float(np.mean(precisions)), float(np.mean(f1_values))


def plot_curves(results, title, save_path):
    plt.figure(figsize=(10, 5))
    for name, result in results.items():
        plt.plot(result["recall_curve"], result["precision_curve"], marker="o", markersize=3,
                 label=f"{name} (AUC = {result['pr_auc']:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def read_seke(name, df, test_docs):
    path = os.path.join(SEKE_RESULTS_FOLDER, f"task10_{name}_seed_42.tsv")
    if not os.path.exists(path):
        print(f"  SEKE file not found, skipped: {path}")
        return None
    seke = pd.read_csv(path, sep="\t", dtype=str).fillna("")
    if len(seke) != len(test_docs):
        raise ValueError(f"{path} has {len(seke)} rows, the test part has {len(test_docs)} documents")
    matches = sum(re.findall(r"[a-z0-9]+", df["abstract"][d].lower())[:3] == re.findall(r"[a-z0-9]+", t.lower())[:3]
                  for t, d in zip(seke["text"], test_docs))
    print(f"  SEKE order check: {matches} of {len(test_docs)} rows start like the expected test document")
    return [split_keywords(cell) for cell in seke["preds"]]


def main():
    tables = {name: pd.read_csv(f"{TABLE_FOLDER}/{name}.csv", dtype=str).fillna("") for name in DATASETS}
    for name, df in tables.items():
        absent = [f"{m}_top20" for m in METHODS if f"{m}_top20" not in df.columns]
        if absent:
            raise ValueError(f"{name}: the extractor columns are missing, run step 2 of task 10 first")

    embeddings = None
    if USE_TASK9_VARIANT:
        embeddings = globals().get("benchmark_embeddings")
        if embeddings is None:
            needed = set()
            for df in tables.values():
                for text in df["abstract"]:
                    needed.update(content_tokens(text))
            print(f"Loading word2vec vectors for {len(needed)} distinct words (takes a few minutes) ...")
            embeddings = load_needed_vectors(MODEL_PATH, needed)
            globals()["benchmark_embeddings"] = embeddings
            print(f"Words found in the word2vec vocabulary: {sum(1 for w in needed if w in embeddings) / len(needed):.1%}")

    summary = []
    for name, df in tables.items():
        print(f"\n===== {name} =====")
        split = df["split"].to_numpy()
        train_docs, test_docs = np.where(split == "train")[0], np.where(split == "test")[0]
        references = [split_keywords(cell) for cell in df["keywords"]]
        test_references = [references[d] for d in test_docs]
        print(f"  {len(train_docs)} training and {len(test_docs)} test documents")

        candidates = build_table(df)
        keys_by_doc = candidates.groupby("doc")["key"].agg(set)
        covered = []
        for d in test_docs:
            keys = {normalize_phrase(r) for r in references[d]} - {""}
            covered.append(len(keys & keys_by_doc.get(d, set())) / max(len(keys), 1))
        print(f"  gold keywords among the candidates of the supervised ranker (recall upper bound): {np.mean(covered):.1%}")
        train, test = candidates[candidates["doc"].isin(train_docs)], candidates[candidates["doc"].isin(test_docs)]

        predictions = {}
        for method in METHODS:
            predictions[method] = [split_keywords(df[f"{method}_keywords"][d]) for d in test_docs]

        print("  training logistic regression and gradient boosting ...")
        logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
        logistic.fit(train[FEATURES], train["label"])
        predictions["logistic regression"] = ranked_phrases(test, logistic.predict_proba(test[FEATURES])[:, 1], test_docs)
        boosting = make_model().fit(train[FEATURES], train["label"])
        predictions["gradient boosting"] = ranked_phrases(test, boosting.predict_proba(test[FEATURES])[:, 1], test_docs)

        if embeddings is not None:
            print("  training the task 9 variant (embedding features, diversity penalty by cross-validation) ...")
            extended = add_embedding_features(candidates, df, embeddings)
            extended_train = extended[extended["doc"].isin(train_docs)]
            extended_test = extended[extended["doc"].isin(test_docs)]
            variant = make_model().fit(extended_train[ALL_FEATURES], extended_train["label"])
            penalty, _ = choose_penalty(extended_train, references)
            scores = variant.predict_proba(extended_test[ALL_FEATURES])[:, 1]
            predictions["task 9 variant"] = ranked_phrases(extended_test, scores, test_docs, penalty)
            print(f"  chosen diversity penalty: {penalty}")

        seke = read_seke(name, df, test_docs)
        if seke is not None:
            predictions["SEKE"] = seke

        results = {method: evaluate_method(pred, test_references) for method, pred in predictions.items()}
        table = make_table(results)
        table["keywords per doc"] = [round(float(np.mean([len(p) for p in predictions[m]])), 2) for m in results]
        strict = [strict_scores(predictions[m], test_references) for m in results]
        table["P@10 strict"] = [round(v[0], 4) for v in strict]
        table["F1@10 strict"] = [round(v[1], 4) for v in strict]
        print(table.to_string(index=False))

        table.to_csv(f"{RESULTS_FOLDER}/metrics_{TAG10}_{name}.csv", index=False)
        plot_curves(results, f"Precision-recall curves ({TAG10}, {name}, test 20%)",
                    f"{RESULTS_FOLDER}/pr_curves_{TAG10}_{name}.png")
        summary.append(table.assign(dataset=name))

    pd.concat(summary).to_csv(f"{RESULTS_FOLDER}/metrics_{TAG10}_summary.csv", index=False)
    print(f"\nAll tables are saved in {RESULTS_FOLDER}/metrics_{TAG10}_*.csv")


if __name__ == "__main__":
    main()